# Milestone 3 — Baseline evaluation (Gemma 4 E4B)

Run on **Colab T4** (~1-2 credits for 500 notes). Upload this repo or mount Drive.

Outputs:
- `predictions_baseline.jsonl`
- Download and score locally: `python scripts/eval.py --predictions predictions_baseline.jsonl`

In [ ]:
!pip install -q transformers accelerate bitsandbytes peft trl scikit-learn

In [ ]:
import os, sys, json, time
from pathlib import Path

# Change if repo is elsewhere
ROOT = Path('/content/finetune')  # or Path('.') if cloned in Colab
sys.path.insert(0, str(ROOT))

from src.prompts import format_for_model
from src.validate import filter_codes, load_whitelists, parse_model_output

GOLD = ROOT / 'data/test.jsonl'
OUT = ROOT / 'predictions_baseline.jsonl'
MODEL_ID = 'google/gemma-4-E4B-it'

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant,
    device_map='auto',
)
model.eval()

In [ ]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

icd_w, cpt_w = load_whitelists(ROOT / 'codes')
rows = load_jsonl(GOLD)
print('Notes:', len(rows))

In [ ]:
def generate(note):
    prompt = format_for_model(tokenizer, note, tokenize=False)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    new = out[0, inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new, skip_special_tokens=True)

preds = []
t0 = time.time()
for i, row in enumerate(rows, 1):
    raw = generate(row['note'])
    parsed = parse_model_output(raw)
    f = filter_codes(parsed, icd_w, cpt_w)
    preds.append({
        'id': row['id'],
        'raw_output': raw,
        'json_valid': f['json_valid'],
        'icd10_pre': f['icd10_pre'],
        'cpt_pre': f['cpt_pre'],
        'icd10': f['icd10'],
        'cpt': f['cpt'],
        'invalid_icd10': f['invalid_icd10'],
        'invalid_cpt': f['invalid_cpt'],
    })
    if i % 25 == 0:
        print(i, '/', len(rows), 'elapsed', int(time.time()-t0), 's')

with open(OUT, 'w') as f:
    for p in preds:
        f.write(json.dumps(p) + '\n')
print('Saved', OUT)

In [ ]:
# Score in notebook (or download OUT and run scripts/eval.py locally)
!python {ROOT}/scripts/eval.py --predictions {OUT} --gold {GOLD} --output {ROOT}/results/baseline.json